# Exercícios de revisão — confira suas previsões

**Programação C (COMP0512) — UFS — 2026.2**

Na aula você respondeu oito exercícios no papel, de recursão a filas, sem compilar nada. Aqui
estão os mesmos programas, prontos para rodar. Para cada rodada:

1. releia a pergunta e a resposta que você escreveu na folha;
2. execute as células;
3. se a saída discordar da sua previsão, volte ao resumo daquela rodada nos slides.

Alguns programas têm defeito **de propósito**: eles vêm compilados com `-fsanitize=address`, que
para o programa no primeiro acesso inválido e diz a linha. Leia o relatório até a primeira linha
que cita um arquivo `.c` — é ali que está o erro.

In [ ]:
# Confira se o gcc está disponível no seu ambiente
!gcc --version | head -1

## R1. Recursão: ida, volta e a árvore de chamadas

O que `f(5)` imprime? Quanto vale `g(4)`, em quantas chamadas, e quantos registros de `g` ficam
na pilha ao mesmo tempo, no máximo?

O programa conta as chamadas com uma variável `static` e mede a altura da pilha somando 1 ao
entrar em `g` e subtraindo 1 ao sair.

In [ ]:
%%writefile r1_recursao.c
/* r1_recursao.c --- o que se imprime na ida e na volta, e quantas chamadas */
#include <stdio.h>

void f(int n)
{
    if (n <= 0)
        return;
    printf("%d ", n);      /* na ida */
    f(n - 2);
    printf("%d ", n);      /* na volta */
}

static int chamadas = 0;   /* quantas vezes g foi chamada */
static int nivel = 0;      /* quantos registros de g estao na pilha agora */
static int pico = 0;       /* o maior valor que nivel atingiu */

int g(int n)
{
    chamadas++;
    nivel++;
    if (nivel > pico)
        pico = nivel;
    int r;
    if (n < 2)
        r = 1;
    else
        r = g(n - 1) + g(n - 2);
    nivel--;
    return r;
}

int main(void)
{
    printf("(a) f(5): ");
    f(5);
    printf("\n");
    int r = g(4);
    printf("(b) g(4) = %d, com %d chamadas\n", r, chamadas);
    printf("(c) no maximo %d registros de g ao mesmo tempo\n", pico);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic r1_recursao.c -o r1 && ./r1

**Tarefa.** Troque `f(n - 2)` por `f(n - 1)` e preveja, no papel, o que `f(4)` imprime. Depois
troque a chamada em `main` e confira. Em seguida, preveja quantas chamadas `g(6)` faz.

## R2. Recursão e memória

**A** guarda o `1 +` em cada registro até a volta; **B** leva o parcial na ida. Com uma pilha de
8 MB e ~48 bytes por registro, `conta(100000)` termina e `conta(1000000)` não.

O programa imprime **B** antes de **A** (com `fflush`), para você ver até onde cada uma chega.

In [ ]:
%%writefile r2_memoria.c
/* r2_memoria.c --- a conta que fica para a volta e o limite da pilha */
#include <stdio.h>
#include <stdlib.h>

/* A: o "1 +" fica pendente em cada registro */
long conta(int n)
{
    if (n == 0)
        return 0;
    return 1 + conta(n - 1);
}

/* B: o parcial desce junto; nada fica para a volta */
long conta_acum(int n, long acc)
{
    if (n == 0)
        return acc;
    return conta_acum(n - 1, acc + 1);
}

int main(int argc, char *argv[])
{
    int n = (argc > 1) ? atoi(argv[1]) : 100000;
    printf("B: conta_acum(%d) = %ld\n", n, conta_acum(n, 0));
    fflush(stdout);
    printf("A: conta(%d) = %ld\n", n, conta(n));
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic r2_memoria.c -o r2
!./r2 100000
!./r2 1000000; echo "codigo de saida: $?"

Um código de saída 139 (ou a mensagem *Segmentation fault*) é o estouro de pilha: o sistema
encerrou o programa. Repare que, **sem otimização**, nem a linha de **B** apareceu: ser de cauda
não basta, é preciso que o compilador reaproveite o registro. Veja o tamanho da pilha desta
máquina:

In [ ]:
!ulimit -s

Agora com `-O2`. O compilador troca a chamada de cauda de **B** por um salto — e, em geral,
consegue reescrever até **A** com um acumulador. Isso é uma **otimização**, não uma garantia da
linguagem: outro compilador, outra versão ou uma função um pouco mais complicada, e a pilha volta
a crescer.

In [ ]:
!gcc -Wall -Wextra -pedantic -O2 r2_memoria.c -o r2_o2 && ./r2_o2 1000000; echo "codigo de saida: $?"

Agora a função **C**, que devolve o endereço de uma variável local. Leia o aviso do compilador
antes de olhar a saída: o que o programa imprime **não** é garantido — pode ser 42 por sorte,
lixo, ou um *segmentation fault* (o gcc costuma trocar o endereço por `NULL`).

In [ ]:
%%writefile r2_local.c
/* r2_local.c --- devolver o endereco de uma variavel local */
#include <stdio.h>

int *dobro(int x)
{
    int r = 2 * x;
    return &r;          /* r morre com o return */
}

int main(void)
{
    int *p = dobro(21);
    printf("%d\n", *p);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic r2_local.c -o r2_local && ./r2_local; echo "codigo de saida: $?"

Três correções que calculam a mesma coisa sem devolver endereço de local:

In [ ]:
%%writefile r2_local_corrigido.c
/* r2_local_corrigido.c --- tres formas de devolver o dobro sem endereco de local */
#include <stdio.h>
#include <stdlib.h>

/* 1) devolver o valor: a copia sobrevive ao return */
int dobro(int x)
{
    return 2 * x;
}

/* 2) o chamador da o lugar onde guardar */
void dobro_em(int x, int *saida)
{
    *saida = 2 * x;
}

/* 3) bloco no heap: quem recebe e dono e precisa dar free */
int *dobro_heap(int x)
{
    int *r = malloc(sizeof *r);
    if (r != NULL)
        *r = 2 * x;
    return r;
}

int main(void)
{
    int a;
    dobro_em(21, &a);
    int *h = dobro_heap(21);
    if (h == NULL)
        return 1;
    printf("%d %d %d\n", dobro(21), a, *h);
    free(h);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic r2_local_corrigido.c -o r2_local_corrigido && ./r2_local_corrigido

**Tarefa.** Descubra, por tentativa (dobrando e depois ajustando), o maior `n` que `./r2 n`
aguenta nesta máquina. Divida o tamanho da pilha (`ulimit -s`, em KB) por esse `n`: quantos bytes
tem cada registro de `conta`?

## R3. Lista encadeada: inserir no início, `sizeof(No)` e liberar

Por que a lista sai `2 -> 8 -> 5`? Quanto vale `sizeof(No)` numa máquina de 64 bits?

In [ ]:
%%writefile r3_lista.c
/* r3_lista.c --- inserir no inicio inverte a ordem */
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

No *insere_inicio(No *ini, int valor)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = valor;
    novo->prox = ini;
    return novo;
}

void imprime(const No *ini)
{
    for (const No *p = ini; p != NULL; p = p->prox)
        printf("%d -> ", p->valor);
    printf("NULL\n");
}

void libera(No *ini)
{
    while (ini != NULL) {
        No *prox = ini->prox;   /* guarda o caminho antes do free */
        free(ini);
        ini = prox;
    }
}

int main(void)
{
    No *lista = NULL;
    int entrada[] = {5, 8, 2};
    for (int i = 0; i < 3; i++)
        lista = insere_inicio(lista, entrada[i]);
    imprime(lista);
    printf("sizeof(No) = %zu\n", sizeof(No));
    libera(lista);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic r3_lista.c -o r3 && ./r3

O laço de liberação do exercício, com o `p = p->prox` rodando **depois** do `free(p)`. Sem
ferramenta, ele quase sempre "funciona"; com o AddressSanitizer, não:

In [ ]:
%%writefile r3_libera_errada.c
/* r3_libera_errada.c --- o laco que le p->prox de um no ja liberado */
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

No *insere_inicio(No *ini, int valor)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = valor;
    novo->prox = ini;
    return novo;
}

int main(void)
{
    No *lista = NULL;
    int entrada[] = {5, 8, 2};
    for (int i = 0; i < 3; i++)
        lista = insere_inicio(lista, entrada[i]);
    for (No *p = lista; p != NULL; p = p->prox)   /* p->prox depois do free */
        free(p);
    printf("liberou (sera?)\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic -g -fsanitize=address r3_libera_errada.c -o r3_libera_errada && ./r3_libera_errada 2>&1 | head -12

**Tarefa.** Corrija o laço de `r3_libera_errada.c` guardando `p->prox` numa variável antes do
`free` (como em `libera`, de `r3_lista.c`) e rode de novo com `-fsanitize=address`: o relatório
tem de sumir.

## R4. Remoção: o meio funciona, as pontas não

A lista é `10 -> 20 -> 30`. O programa recebe na linha de comando o valor a remover.

In [ ]:
%%writefile r4_remove.c
/* r4_remove.c --- uma remocao que funciona so no caso do meio */
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

No *insere_inicio(No *ini, int valor)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = valor;
    novo->prox = ini;
    return novo;
}

No *remove_valor(No *ini, int x)
{
    No *ant = NULL, *p = ini;
    while (p->valor != x && p != NULL) {
        ant = p;
        p = p->prox;
    }
    ant->prox = p->prox;
    free(p);
    return ini;
}

void imprime(const No *ini)
{
    for (const No *p = ini; p != NULL; p = p->prox)
        printf("%d -> ", p->valor);
    printf("NULL\n");
}

void libera(No *ini)
{
    while (ini != NULL) {
        No *prox = ini->prox;
        free(ini);
        ini = prox;
    }
}

int main(int argc, char *argv[])
{
    int x = (argc > 1) ? atoi(argv[1]) : 20;
    No *lista = NULL;
    for (int v = 30; v >= 10; v -= 10)       /* 10 -> 20 -> 30 */
        lista = insere_inicio(lista, v);
    imprime(lista);
    printf("remove_valor(lista, %d)\n", x);
    fflush(stdout);
    lista = remove_valor(lista, x);
    imprime(lista);
    libera(lista);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic r4_remove.c -o r4
!./r4 20
!./r4 10; echo "codigo de saida: $?"
!./r4 99; echo "codigo de saida: $?"

Só o 20 funciona. Para o 10, `ant` ainda é `NULL` na linha `ant->prox = p->prox`; para o 99,
`p` chega a `NULL` e `p->valor` é lido antes do teste `p != NULL`. A versão corrigida trata os
três casos e mostra também a remoção com ponteiro para ponteiro, que não precisa de caso especial:

In [ ]:
%%writefile r4_remove_corrigido.c
/* r4_remove_corrigido.c --- os tres casos: meio, primeiro no e ausente */
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *prox;
} No;

No *insere_inicio(No *ini, int valor)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = valor;
    novo->prox = ini;
    return novo;
}


No *remove_valor(No *ini, int x)
{
    No *ant = NULL, *p = ini;
    while (p != NULL && p->valor != x) {   /* testa NULL antes de ler */
        ant = p;
        p = p->prox;
    }
    if (p == NULL)                         /* nao achou: nada muda */
        return ini;
    if (ant == NULL)                       /* era o primeiro: muda o inicio */
        ini = p->prox;
    else
        ant->prox = p->prox;
    free(p);
    return ini;
}

/* a mesma remocao com ponteiro para ponteiro: sem caso especial */
void remove_valor_pp(No **pp, int x)
{
    while (*pp != NULL && (*pp)->valor != x)
        pp = &(*pp)->prox;
    if (*pp != NULL) {
        No *p = *pp;
        *pp = p->prox;
        free(p);
    }
}

void imprime(const No *ini)
{
    for (const No *p = ini; p != NULL; p = p->prox)
        printf("%d -> ", p->valor);
    printf("NULL\n");
}

void libera(No *ini)
{
    while (ini != NULL) {
        No *prox = ini->prox;
        free(ini);
        ini = prox;
    }
}

int main(int argc, char *argv[])
{
    int x = (argc > 1) ? atoi(argv[1]) : 20;
    No *lista = NULL;
    for (int v = 30; v >= 10; v -= 10)       /* 10 -> 20 -> 30 */
        lista = insere_inicio(lista, v);
    imprime(lista);
    printf("remove_valor(lista, %d)\n", x);
    fflush(stdout);
    lista = remove_valor(lista, x);
    imprime(lista);
    remove_valor_pp(&lista, 30);
    printf("remove_valor_pp(&lista, 30)\n");
    imprime(lista);
    libera(lista);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic r4_remove_corrigido.c -o r4_corrigido
!./r4_corrigido 10
!./r4_corrigido 99

**Tarefa.** Escreva `No *insere_ordenado(No *ini, int x)` sem ponteiro para ponteiro, parando no
nó **anterior**, e teste inserindo 5, 25 e 35 em `10 -> 20 -> 30`. Quais desses três casos
mudam o início?

## R5. Lista dupla: quatro ponteiros e o `fim` esquecido

O programa encaixa o 25 depois do 20 (`insere_depois`, as quatro linhas do exercício), remove o
20 do meio e, por fim, remove o **último** nó. A `remove_no` deste arquivo é a do exercício: não
atualiza `l->fim`.

In [ ]:
%%writefile r5_dupla.c
/* r5_dupla.c --- uma remove_no que esquece um dos extremos */
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *ant;
    struct no *prox;
} No;

typedef struct {
    No *ini;
    No *fim;
} Lista;

/* coloca x no fim da lista */
void insere_fim(Lista *l, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = x;
    novo->prox = NULL;
    novo->ant = l->fim;
    if (l->fim == NULL)
        l->ini = novo;
    else
        l->fim->prox = novo;
    l->fim = novo;
}

/* encaixa x logo depois do no p (p nao e o ultimo) */
void insere_depois(No *p, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = x;
    novo->ant = p;               /* 1 */
    novo->prox = p->prox;        /* 2 */
    p->prox->ant = novo;         /* 3 */
    p->prox = novo;              /* 4 */
}

void remove_no(Lista *l, No *p)
{
    if (p->ant == NULL)
        l->ini = p->prox;
    else
        p->ant->prox = p->prox;
    if (p->prox != NULL)
        p->prox->ant = p->ant;
    free(p);
}

void imprime(const Lista *l)
{
    printf("ida:  ");
    for (const No *p = l->ini; p != NULL; p = p->prox)
        printf(" %d", p->valor);
    printf("\nvolta:");
    for (const No *p = l->fim; p != NULL; p = p->ant)
        printf(" %d", p->valor);
    printf("\n");
}

void libera(Lista *l)
{
    No *p = l->ini;
    while (p != NULL) {
        No *prox = p->prox;
        free(p);
        p = prox;
    }
    l->ini = l->fim = NULL;
}

int main(void)
{
    Lista l = {NULL, NULL};
    for (int v = 10; v <= 30; v += 10)
        insere_fim(&l, v);
    insere_depois(l.ini->prox, 25);      /* entre o 20 e o 30 */
    imprime(&l);
    remove_no(&l, l.ini->prox);          /* (b) tira o 20, do meio */
    imprime(&l);
    remove_no(&l, l.fim);                /* (c) tira o 30, o ultimo */
    imprime(&l);
    libera(&l);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic -g -fsanitize=address r5_dupla.c -o r5 && ./r5 2>&1 | head -16

Repare onde o AddressSanitizer acusa o erro: na função `imprime`, ao percorrer **de trás para
frente** a partir de `l->fim` — e não em `remove_no`, onde ele foi cometido. Com a correção
(o `if` simétrico ao do início), as três impressões saem certas:

In [ ]:
%%writefile r5_dupla_corrigido.c
/* r5_dupla_corrigido.c --- remove_no cuidando dos dois extremos */
#include <stdio.h>
#include <stdlib.h>

typedef struct no {
    int valor;
    struct no *ant;
    struct no *prox;
} No;

typedef struct {
    No *ini;
    No *fim;
} Lista;

/* coloca x no fim da lista */
void insere_fim(Lista *l, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = x;
    novo->prox = NULL;
    novo->ant = l->fim;
    if (l->fim == NULL)
        l->ini = novo;
    else
        l->fim->prox = novo;
    l->fim = novo;
}

/* encaixa x logo depois do no p (p nao e o ultimo) */
void insere_depois(No *p, int x)
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = x;
    novo->ant = p;               /* 1 */
    novo->prox = p->prox;        /* 2 */
    p->prox->ant = novo;         /* 3 */
    p->prox = novo;              /* 4 */
}

void remove_no(Lista *l, No *p)
{
    if (p->ant == NULL)
        l->ini = p->prox;
    else
        p->ant->prox = p->prox;
    if (p->prox == NULL)         /* era o ultimo: muda o fim */
        l->fim = p->ant;
    else
        p->prox->ant = p->ant;
    free(p);
}

void imprime(const Lista *l)
{
    printf("ida:  ");
    for (const No *p = l->ini; p != NULL; p = p->prox)
        printf(" %d", p->valor);
    printf("\nvolta:");
    for (const No *p = l->fim; p != NULL; p = p->ant)
        printf(" %d", p->valor);
    printf("\n");
}

void libera(Lista *l)
{
    No *p = l->ini;
    while (p != NULL) {
        No *prox = p->prox;
        free(p);
        p = prox;
    }
    l->ini = l->fim = NULL;
}

int main(void)
{
    Lista l = {NULL, NULL};
    for (int v = 10; v <= 30; v += 10)
        insere_fim(&l, v);
    insere_depois(l.ini->prox, 25);      /* entre o 20 e o 30 */
    imprime(&l);
    remove_no(&l, l.ini->prox);          /* (b) tira o 20, do meio */
    imprime(&l);
    remove_no(&l, l.fim);                /* (c) tira o 30, o ultimo */
    imprime(&l);
    libera(&l);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic r5_dupla_corrigido.c -o r5_corrigido && ./r5_corrigido

**Tarefa.** Em `insere_depois`, troque a ordem das linhas marcadas `/* 3 */` e `/* 4 */`, rode
`r5_corrigido` e compare a impressão de ida com a de volta. Explique a diferença desenhando os
ponteiros.

## R6. Pilhas: rastreio, balanceamento e crescimento

(a) o rastreio do exercício; (b) a verificação de balanceamento, agora dizendo **em que posição**
o erro foi notado (a posição igual ao tamanho da entrada quer dizer "sobrou algum aberto no
fim"); (c) quantas vezes o vetor cresce ao empilhar 100 valores, dobrando ou somando 4.

In [ ]:
%%writefile r6_pilha.c
/* r6_pilha.c --- rastreio de uma pilha, balanceamento e crescimento do vetor */
#include <stdio.h>
#include <stdlib.h>

typedef struct {
    int *v;      /* v[0..n-1]: o topo e v[n-1] */
    int n;
    int cap;
    int cresceu; /* quantas vezes o vetor aumentou */
} Pilha;

static void *aloca(void *antigo, size_t tam)
{
    void *p = realloc(antigo, tam);
    if (p == NULL) {
        perror("realloc");
        exit(1);
    }
    return p;
}

Pilha *pilha_cria(void)
{
    Pilha *p = aloca(NULL, sizeof *p);
    p->n = 0;
    p->cap = 4;
    p->cresceu = 0;
    p->v = aloca(NULL, p->cap * sizeof *p->v);
    return p;
}

void pilha_libera(Pilha *p)
{
    free(p->v);
    free(p);
}

/* passo = 0: dobra a capacidade; passo > 0: soma passo */
void empilha_com(Pilha *p, int x, int passo)
{
    if (p->n == p->cap) {
        p->cap = (passo == 0) ? 2 * p->cap : p->cap + passo;
        p->v = aloca(p->v, p->cap * sizeof *p->v);
        p->cresceu++;
    }
    p->v[p->n] = x;
    p->n++;
}

void empilha(Pilha *p, int x)
{
    empilha_com(p, x, 0);
}

int desempilha(Pilha *p)
{
    if (p->n == 0) {
        fprintf(stderr, "desempilha: pilha vazia\n");
        exit(1);
    }
    p->n--;
    return p->v[p->n];
}

int topo(const Pilha *p)
{
    if (p->n == 0) {
        fprintf(stderr, "topo: pilha vazia\n");
        exit(1);
    }
    return p->v[p->n - 1];
}

int vazia(const Pilha *p)
{
    return p->n == 0;
}

/* -1 se balanceada; senao, a posicao onde o erro foi notado
   (o tamanho de s, se sobrou algum aberto) */
int primeiro_erro(const char *s)
{
    Pilha *p = pilha_cria();
    int i, erro = -1;
    for (i = 0; s[i] != '\0' && erro < 0; i++) {
        char c = s[i];
        if (c == '(' || c == '[' || c == '{')
            empilha(p, c);
        else if (c == ')' || c == ']' || c == '}') {
            if (vazia(p)) {
                erro = i;
            } else {
                char a = (char) desempilha(p);
                if (!((a == '(' && c == ')') || (a == '[' && c == ']') ||
                      (a == '{' && c == '}')))
                    erro = i;
            }
        }
    }
    if (erro < 0 && !vazia(p))
        erro = i;
    pilha_libera(p);
    return erro;
}

int main(void)
{
    /* (a) */
    Pilha *p = pilha_cria();
    for (int i = 1; i <= 5; i++)
        empilha(p, 10 * i);
    int a = desempilha(p);
    int b = desempilha(p);
    empilha(p, a - b);
    printf("(a) topo: %d\n", topo(p));
    printf("    esvaziando:");
    while (!vazia(p))
        printf(" %d", desempilha(p));
    printf("\n");
    pilha_libera(p);

    /* (b) */
    const char *testes[] = {"{[()()]}", "[(])", "{}("};
    for (int i = 0; i < 3; i++) {
        int e = primeiro_erro(testes[i]);
        if (e < 0)
            printf("(b) %-10s balanceada\n", testes[i]);
        else
            printf("(b) %-10s erro notado na posicao %d\n", testes[i], e);
    }

    /* (c) */
    int passos[] = {0, 4};
    for (int k = 0; k < 2; k++) {
        p = pilha_cria();
        for (int i = 0; i < 100; i++)
            empilha_com(p, i, passos[k]);
        printf("(c) %s: cresceu %d vezes, cap final %d\n",
               passos[k] == 0 ? "dobrando " : "somando 4", p->cresceu, p->cap);
        pilha_libera(p);
    }
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic r6_pilha.c -o r6 && ./r6

**Tarefa.** Acrescente ao vetor `testes` as entradas `")("`, `"{[}]"` e `"a(b[c]d)e"`, preveja
o resultado de cada uma e rode. Depois, troque os 100 valores de (c) por 1000: quantas vezes
cada estratégia cresce agora?

## R7. Filas: o anel, desenrolado ou não

A sequência do exercício numa fila circular com `cap = 4`. O programa mostra o vetor inteiro,
inclusive as posições que não fazem parte da fila (as posições vazias começam em 0 só para o
desenho sair limpo).

In [ ]:
%%writefile r7_fila.c
/* r7_fila.c --- fila circular com ini e n: onde cada valor vai parar */
#include <stdio.h>
#include <stdlib.h>

typedef struct {
    int *v;    /* a frente e v[ini]; os n seguintes, dando a volta */
    int ini;
    int n;
    int cap;
} Fila;

static void *aloca(void *antigo, size_t tam)
{
    void *p = realloc(antigo, tam);
    if (p == NULL) {
        perror("realloc");
        exit(1);
    }
    return p;
}

Fila *fila_cria(void)
{
    Fila *f = aloca(NULL, sizeof *f);
    f->ini = 0;
    f->n = 0;
    f->cap = 4;
    f->v = aloca(NULL, f->cap * sizeof *f->v);
    for (int i = 0; i < f->cap; i++)
        f->v[i] = 0;          /* so para o desenho sair limpo */
    return f;
}

void fila_libera(Fila *f)
{
    free(f->v);
    free(f);
}

/* novo vetor com o dobro, ja na ordem: a frente vai para v[0] */
static void cresce(Fila *f)
{
    int *novo = aloca(NULL, 2 * f->cap * sizeof *novo);
    for (int i = 0; i < 2 * f->cap; i++)
        novo[i] = 0;
    for (int i = 0; i < f->n; i++)
        novo[i] = f->v[(f->ini + i) % f->cap];
    free(f->v);
    f->v = novo;
    f->ini = 0;
    f->cap *= 2;
}

void enfileira(Fila *f, int x)
{
    if (f->n == f->cap)
        cresce(f);
    f->v[(f->ini + f->n) % f->cap] = x;
    f->n++;
}

int desenfileira(Fila *f)
{
    if (f->n == 0) {
        fprintf(stderr, "desenfileira: fila vazia\n");
        exit(1);
    }
    int x = f->v[f->ini];
    f->ini = (f->ini + 1) % f->cap;
    f->n--;
    return x;
}

void mostra(const char *rotulo, const Fila *f)
{
    printf("%s v = [", rotulo);
    for (int i = 0; i < f->cap; i++)
        printf(i ? " %d" : "%d", f->v[i]);
    printf("]  ini = %d  n = %d  cap = %d\n", f->ini, f->n, f->cap);
}

int main(void)
{
    Fila *f = fila_cria();
    enfileira(f, 1);
    enfileira(f, 2);
    enfileira(f, 3);
    desenfileira(f);
    desenfileira(f);
    enfileira(f, 4);
    enfileira(f, 5);
    mostra("(a)", f);
    printf("    o proximo enfileira escreve em v[%d]\n",
           (f->ini + f->n) % f->cap);
    enfileira(f, 6);
    mostra("(b) depois do 6:", f);
    enfileira(f, 7);
    mostra("(b) depois do 7:", f);
    printf("    saem:");
    while (f->n > 0)
        printf(" %d", desenfileira(f));
    printf("\n");
    fila_libera(f);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic r7_fila.c -o r7 && ./r7

Agora a mesma fila crescendo com um simples `realloc`, como na pilha. As posições novas são
marcadas com `-1`, para que se veja de onde sai o lixo:

In [ ]:
%%writefile r7_dobra_errada.c
/* r7_dobra_errada.c --- crescer a fila circular com realloc, como na pilha */
#include <stdio.h>
#include <stdlib.h>

typedef struct {
    int *v;    /* a frente e v[ini]; os n seguintes, dando a volta */
    int ini;
    int n;
    int cap;
} Fila;

static void *aloca(void *antigo, size_t tam)
{
    void *p = realloc(antigo, tam);
    if (p == NULL) {
        perror("realloc");
        exit(1);
    }
    return p;
}

Fila *fila_cria(void)
{
    Fila *f = aloca(NULL, sizeof *f);
    f->ini = 0;
    f->n = 0;
    f->cap = 4;
    f->v = aloca(NULL, f->cap * sizeof *f->v);
    for (int i = 0; i < f->cap; i++)
        f->v[i] = 0;          /* so para o desenho sair limpo */
    return f;
}

void fila_libera(Fila *f)
{
    free(f->v);
    free(f);
}

/* ERRADO: dobra no lugar, sem desenrolar o anel */
static void cresce(Fila *f)
{
    f->v = aloca(f->v, 2 * f->cap * sizeof *f->v);
    for (int i = f->cap; i < 2 * f->cap; i++)
        f->v[i] = -1;         /* marca as posicoes novas, que ninguem preencheu */
    f->cap *= 2;
}

void enfileira(Fila *f, int x)
{
    if (f->n == f->cap)
        cresce(f);
    f->v[(f->ini + f->n) % f->cap] = x;
    f->n++;
}

int desenfileira(Fila *f)
{
    if (f->n == 0) {
        fprintf(stderr, "desenfileira: fila vazia\n");
        exit(1);
    }
    int x = f->v[f->ini];
    f->ini = (f->ini + 1) % f->cap;
    f->n--;
    return x;
}

void mostra(const char *rotulo, const Fila *f)
{
    printf("%s v = [", rotulo);
    for (int i = 0; i < f->cap; i++)
        printf(i ? " %d" : "%d", f->v[i]);
    printf("]  ini = %d  n = %d  cap = %d\n", f->ini, f->n, f->cap);
}

int main(void)
{
    Fila *f = fila_cria();
    enfileira(f, 1);
    enfileira(f, 2);
    enfileira(f, 3);
    desenfileira(f);
    desenfileira(f);
    enfileira(f, 4);
    enfileira(f, 5);
    mostra("(a)", f);
    printf("    o proximo enfileira escreve em v[%d]\n",
           (f->ini + f->n) % f->cap);
    enfileira(f, 6);
    mostra("(b) depois do 6:", f);
    enfileira(f, 7);
    mostra("(b) depois do 7:", f);
    printf("    saem:");
    while (f->n > 0)
        printf(" %d", desenfileira(f));
    printf("\n");
    fila_libera(f);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -pedantic r7_dobra_errada.c -o r7_dobra_errada && ./r7_dobra_errada

**Tarefa.** Em `r7_fila.c`, troque a sequência de `main` para que a fila encha **sem** ter dado
a volta (`ini = 0`) e cresça. Nesse caso o `realloc` simples também funcionaria? Por quê?

## R8. Síntese

A rodada 8 não tem programa: confira a tabela de custos e as escolhas de estrutura no gabarito
dos slides. Para fixar, responda sem olhar:

- Por que a fila numa lista **sai do início e entra no fim**, e não o contrário?
- Qual estrutura das aulas faria `desfazer` *e* `refazer` num editor?
- Em qual das rodadas de hoje uma pilha apareceu sem que ninguém a declarasse?